In [ ]:
# Clone repo AutoShot chính thức
# Repo: https://github.com/wentaozhu/AutoShot
!git clone https://github.com/wentaozhu/AutoShot.git /kaggle/working/AutoShot

# Cài các thư viện cần thiết.
# ffmpeg-python dùng để đọc video trong utils.py của repo.
# einops được model AutoShot import trong file supernet_flattransf_*.py.
!pip install -q ffmpeg-python einops opencv-python tqdm

# AutoShot trích keyframes

## Mô tả
Ở đây, chúng tôi trích xuất hết 96 videos của folder `Video_L30_a` sử dụng AutoShot

In [ ]:
import os
from pathlib import Path

# Thư mục repo AutoShot vừa clone
REPO_DIR = Path("/kaggle/working/AutoShot")

# Đổi path này theo dataset video của bạn trên Kaggle
VIDEO_ROOT = Path("/kaggle/input/datasets/aresusayhi/ai-challenge-2025/Videos/Videos/Video_L30_a/video")

# Đổi path này theo nơi bạn attach checkpoint AutoShot
CKPT_PATH = Path("/kaggle/input/models/khngxuninh/autoshot/pytorch/default/1/ckpt_0_200_0.pth")

print("Repo exists:", REPO_DIR.exists())
print("Video root exists:", VIDEO_ROOT.exists())
print("Checkpoint exists:", CKPT_PATH.exists())

# Liệt kê nhanh vài video tìm được
video_exts = {".mp4", ".avi", ".mov", ".mkv", ".webm"}

videos = [
    p for p in VIDEO_ROOT.rglob("*")
    if p.suffix.lower() in video_exts
]

print("Number of videos:", len(videos))
print("First 5 videos:")
for p in videos[:5]:
    print(" -", p)

In [ ]:
%%writefile /kaggle/working/autoshot_extract_frames.py
"""
Run AutoShot on custom videos and extract 3 representative frames per shot:
first frame, middle frame, last frame.

Input:
    --video_root: folder containing your videos
    --ckpt: path to AutoShot checkpoint ckpt_0_200_0.pth

Output:
    --out_dir/
        shot_segments.csv
        frames/
            video_name/
                shot_0000_first.jpg
                shot_0000_middle.jpg
                shot_0000_last.jpg
                ...
"""

import os
import sys
import csv
import argparse
from pathlib import Path

import cv2
import numpy as np
import torch
from tqdm import tqdm


# ============================================================
# 1. Import AutoShot code
# ============================================================
# Repo AutoShot có các file:
# - utils.py
# - supernet_flattransf_3_8_8_8_13_12_0_16_60.py
# - linear.py
#
# Ta thêm repo_dir vào sys.path để Python import được các file đó.
def add_repo_to_path(repo_dir: str):
    repo_dir = str(Path(repo_dir).resolve())
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)


# ============================================================
# 2. Load AutoShot model
# ============================================================
def load_autoshot_model(repo_dir: str, ckpt_path: str, device: str):
    """
    Load AutoShot architecture + pretrained checkpoint.

    Model class:
        TransNetV2Supernet

    Checkpoint chính thức theo README:
        ckpt_0_200_0.pth

    Trong script gốc của AutoShot, checkpoint được load bằng:
        pretrained_dict = torch.load(pretrained_path)
        pretrained_dict = pretrained_dict["net"]

    Vì vậy ở đây ta hỗ trợ cả 2 trường hợp:
        1. checkpoint có key "net"
        2. checkpoint trực tiếp là state_dict
    """
    add_repo_to_path(repo_dir)

    from supernet_flattransf_3_8_8_8_13_12_0_16_60 import TransNetV2Supernet

    model = TransNetV2Supernet().eval()

    ckpt = torch.load(ckpt_path, map_location=device)

    # Một số checkpoint lưu dict {"net": state_dict}
    # Một số checkpoint khác có thể lưu trực tiếp state_dict
    if isinstance(ckpt, dict) and "net" in ckpt:
        pretrained_state = ckpt["net"]
    else:
        pretrained_state = ckpt

    # Chỉ load những weight có tên trùng với model hiện tại
    # để tránh lỗi nếu checkpoint chứa thêm metadata.
    model_state = model.state_dict()
    matched_state = {
        k: v for k, v in pretrained_state.items()
        if k in model_state and v.shape == model_state[k].shape
    }

    print(f"Model params: {len(model_state)}")
    print(f"Matched checkpoint params: {len(matched_state)}")

    model_state.update(matched_state)
    model.load_state_dict(model_state)

    model = model.to(device)
    model.eval()
    return model


# ============================================================
# 3. Predict shot-boundary score cho từng frame
# ============================================================
@torch.no_grad()
def predict_boundary_scores(model, video_path: str, repo_dir: str, device: str):
    """
    Trả về:
        scores: numpy array shape [num_frames]
                scores[i] là xác suất frame i là shot boundary.

    Logic theo code gốc của AutoShot:
        frames = get_frames(video)
        for batch in get_batches(frames):
            one_hot = predict(batch)
            predictions.append(one_hot[25:75])
        predictions = concat(...)[:len(frames)]

    Vì get_batches() tạo batch 100 frames và chỉ lấy 50 frame giữa,
    nên đoạn [25:75] giúp tránh vùng padding/biên không ổn định.
    """
    add_repo_to_path(repo_dir)
    from utils import get_frames, get_batches

    # get_frames đọc video bằng ffmpeg, resize về 48x27 RGB.
    # Đây là input resolution mà AutoShot code sử dụng.
    frames = get_frames(video_path)

    if len(frames) == 0:
        raise RuntimeError(f"Cannot read frames from video: {video_path}")

    all_scores = []

    for batch in get_batches(frames):
        # batch ban đầu có shape:
        #     [T, H, W, C]
        # với T = 100, H = 27, W = 48, C = 3
        #
        # Model PyTorch cần shape:
        #     [B, C, T, H, W]
        # nên ta transpose rồi thêm batch dimension B=1.
        x = batch.transpose((3, 0, 1, 2))      # [C, T, H, W]
        x = x[np.newaxis, ...]                 # [1, C, T, H, W]
        x = torch.from_numpy(x).float().to(device)

        # Model trả về one_hot hoặc tuple(one_hot, many_hot).
        # one_hot là logit boundary cho từng frame.
        output = model(x)

        if isinstance(output, tuple):
            one_hot_logits = output[0]
        else:
            one_hot_logits = output

        # one_hot_logits shape thường là [1, 100, 1]
        # Lấy batch đầu tiên, sigmoid để chuyển logit -> probability.
        prob = torch.sigmoid(one_hot_logits[0]).detach().cpu().numpy()

        # Squeeze để thành shape [100]
        prob = np.squeeze(prob)

        # Giữ 50 frame giữa, giống code inference gốc.
        all_scores.append(prob[25:75])

    scores = np.concatenate(all_scores, axis=0)[:len(frames)]
    return scores


# ============================================================
# 4. Chuyển boundary frames -> shot segments
# ============================================================
def boundaries_to_shots(boundary_frames, num_frames: int, min_shot_len: int = 5):
    """
    Convert danh sách frame boundary thành danh sách shot [start, end].

    Ví dụ:
        num_frames = 100
        boundary_frames = [20, 60]

    Ta tạo shots:
        [0, 20]
        [21, 60]
        [61, 99]

    min_shot_len:
        Loại bỏ các shot quá ngắn do model detect nhiễu.
        Ví dụ min_shot_len=5 nghĩa là shot phải có ít nhất 5 frames.
    """
    boundary_frames = sorted(set(int(x) for x in boundary_frames))

    shots = []
    start = 0

    for b in boundary_frames:
        # Đảm bảo boundary nằm trong video
        b = max(0, min(b, num_frames - 1))

        end = b

        # Chỉ nhận shot nếu đủ dài
        if end - start + 1 >= min_shot_len:
            shots.append((start, end))

        # Shot tiếp theo bắt đầu sau boundary
        start = b + 1

    # Shot cuối cùng
    if start <= num_frames - 1:
        end = num_frames - 1
        if end - start + 1 >= min_shot_len:
            shots.append((start, end))

    # Nếu vì lý do nào đó không có shot nào, coi cả video là 1 shot
    if len(shots) == 0:
        shots = [(0, num_frames - 1)]

    return shots


# ============================================================
# 5. Đọc frame gốc bằng OpenCV và lưu first/middle/last
# ============================================================
def save_frame_at(cap, frame_idx: int, out_path: Path):
    """
    Lưu frame gốc tại frame_idx ra ảnh jpg.

    Lưu ý:
        AutoShot predict trên frame resize 48x27.
        Nhưng representative frames nên lấy từ video gốc
        để giữ chất lượng cao hơn.
    """
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame_bgr = cap.read()

    if not ok or frame_bgr is None:
        return False

    out_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out_path), frame_bgr)
    return True


def extract_representative_frames(video_path: str, shots, out_video_dir: Path):
    """
    Với mỗi shot [start, end], lưu 3 frame:
        - first  = start
        - middle = (start + end) // 2
        - last   = end

    Trả về list metadata để ghi CSV.
    """
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise RuntimeError(f"OpenCV cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    rows = []

    for shot_id, (start, end) in enumerate(shots):
        middle = (start + end) // 2

        frame_items = [
            ("first", start),
            ("middle", middle),
            ("last", end),
        ]

        for frame_type, frame_idx in frame_items:
            out_name = f"shot_{shot_id:04d}_{frame_type}_f{frame_idx:06d}.jpg"
            out_path = out_video_dir / out_name

            saved = save_frame_at(cap, frame_idx, out_path)

            rows.append({
                "shot_id": shot_id,
                "shot_start_frame": start,
                "shot_end_frame": end,
                "shot_start_sec": start / fps if fps and fps > 0 else None,
                "shot_end_sec": end / fps if fps and fps > 0 else None,
                "frame_type": frame_type,
                "frame_idx": frame_idx,
                "frame_sec": frame_idx / fps if fps and fps > 0 else None,
                "image_path": str(out_path),
                "saved": saved,
                "fps": fps,
                "total_frames_opencv": total_frames,
            })

    cap.release()
    return rows


# ============================================================
# 6. Main pipeline
# ============================================================
def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--repo_dir", type=str, required=True)
    parser.add_argument("--video_root", type=str, required=True)
    parser.add_argument("--ckpt", type=str, required=True)
    parser.add_argument("--out_dir", type=str, required=True)

    # Threshold 0.296 là threshold best F1 được ghi trong script gốc.
    # Bạn có thể tăng nếu muốn ít boundary hơn, giảm nếu muốn nhạy hơn.
    parser.add_argument("--threshold", type=float, default=0.296)

    # Loại bỏ shot quá ngắn do detect nhiễu.
    parser.add_argument("--min_shot_len", type=int, default=5)

    args = parser.parse_args()

    repo_dir = Path(args.repo_dir)
    video_root = Path(args.video_root)
    ckpt_path = Path(args.ckpt)
    out_dir = Path(args.out_dir)

    frames_dir = out_dir / "frames"
    out_dir.mkdir(parents=True, exist_ok=True)
    frames_dir.mkdir(parents=True, exist_ok=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", device)

    model = load_autoshot_model(
        repo_dir=str(repo_dir),
        ckpt_path=str(ckpt_path),
        device=device,
    )

    video_exts = {".mp4", ".avi", ".mov", ".mkv", ".webm"}
    videos = [
        p for p in video_root.rglob("*")
        if p.suffix.lower() in video_exts
    ]

    print(f"Found {len(videos)} videos")

    csv_path = out_dir / "shot_segments.csv"

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        fieldnames = [
            "video_name",
            "video_path",
            "shot_id",
            "shot_start_frame",
            "shot_end_frame",
            "shot_start_sec",
            "shot_end_sec",
            "frame_type",
            "frame_idx",
            "frame_sec",
            "image_path",
            "boundary_threshold",
            "saved",
            "fps",
            "total_frames_opencv",
        ]

        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for video_path in tqdm(videos, desc="Processing videos"):
            video_name = video_path.stem

            try:
                # 1. Predict boundary score cho từng frame
                scores = predict_boundary_scores(
                    model=model,
                    video_path=str(video_path),
                    repo_dir=str(repo_dir),
                    device=device,
                )

                # 2. Lấy các frame có score vượt threshold
                boundary_frames = np.where(scores > args.threshold)[0]

                # 3. Convert boundary -> shot segments
                shots = boundaries_to_shots(
                    boundary_frames=boundary_frames,
                    num_frames=len(scores),
                    min_shot_len=args.min_shot_len,
                )

                # 4. Tạo folder output riêng cho từng video
                safe_video_name = video_name.replace("/", "_").replace(" ", "_")
                out_video_dir = frames_dir / safe_video_name

                # 5. Extract first/middle/last frame cho mỗi shot
                rows = extract_representative_frames(
                    video_path=str(video_path),
                    shots=shots,
                    out_video_dir=out_video_dir,
                )

                # 6. Ghi metadata ra CSV
                for row in rows:
                    row["video_name"] = video_path.name
                    row["video_path"] = str(video_path)
                    row["boundary_threshold"] = args.threshold
                    writer.writerow(row)

            except Exception as e:
                print(f"[ERROR] {video_path}: {e}")

    print("Done!")
    print("CSV saved to:", csv_path)
    print("Frames saved to:", frames_dir)


if __name__ == "__main__":
    main()

In [ ]:
!python /kaggle/working/autoshot_extract_frames.py \
  --repo_dir /kaggle/working/AutoShot \
  --video_root /kaggle/input/datasets/aresusayhi/ai-challenge-2025/Videos/Videos/Video_L30_a/video \
  --ckpt /kaggle/input/models/khngxuninh/autoshot/pytorch/default/1/ckpt_0_200_0.pth \
  --out_dir /kaggle/working/autoshot_output \
  --threshold 0.296 \
  --min_shot_len 5

In [ ]:
import pandas as pd

csv_path = "/kaggle/working/autoshot_output/shot_segments.csv"

df = pd.read_csv(csv_path)
df.head(20)

In [ ]:
df.tail(20)


In [ ]:
len(pd.unique(df["video_path"]))

In [ ]:
import shutil
from pathlib import Path

# Folder cần nén
SOURCE_DIR = Path("/kaggle/working/autoshot_output")

# Tên file zip output, không cần thêm .zip ở đây
ZIP_BASE = Path("/kaggle/working/autoshot_output")

# File zip cuối cùng sẽ là:
ZIP_PATH = Path(str(ZIP_BASE) + ".zip")

# Xóa zip cũ nếu đã tồn tại
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
    print("Đã xóa zip cũ:", ZIP_PATH)

# Nén folder
shutil.make_archive(
    base_name=str(ZIP_BASE),
    format="zip",
    root_dir=str(SOURCE_DIR)
)

print("Đã nén xong:")
print(ZIP_PATH)
print(f"Dung lượng: {ZIP_PATH.stat().st_size / (1024**2):.2f} MB")

In [ ]:
import json

meta_path = "/kaggle/working/my_dataset/dataset-metadata.json"

with open(meta_path, "r") as f:
    meta = json.load(f)

meta["title"] = "AutoShot Output"
meta["id"] = "khngxuninh/autoshot-output"

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(json.dumps(meta, indent=2))

In [ ]:
!kaggle datasets create -p /kaggle/working/my_dataset --dir-mode zip